In [3]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")

💻 Local Lab Server Environment Loaded.


## 1. Load Data & Base Model

In [4]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")

# Load filtered validation sets
val_filtered_full = load_jsonl(f"{DATA_DIR}/val_filtered_top4_BAAI-bge-small-en-v1.5_fullInfo.jsonl")
val_filtered_struct = load_jsonl(f"{DATA_DIR}/val_filtered_top4_BAAI-bge-small-en-v1.5_structOnly.jsonl")
print(f"Loaded {len(val_filtered_full)} filtered full-info and {len(val_filtered_struct)} filtered structural samples.")


Loaded 2039 full-info validation samples and 2039 structural validation samples.
Loaded 2020 filtered full-info and 2020 filtered structural samples.


In [5]:
# === CONFIGURATION ===
COMPUTE_DTYPE = torch.bfloat16

MODEL_ID = "ibm-granite/granite-3.1-2b-instruct"

SFT_STRUCT_DIR = f"{MODELS_DIR}/granite-3.1-2b-instruct_structOnly_LoRA"
SFT_FULL_DIR   = f"{MODELS_DIR}/granite-3.1-2b-instruct_fullInfo_LoRA"

In [6]:
# === LOAD BASE MODEL ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

if MODEL_ID == "mistralai/Mistral-Nemo-Instruct-2407":
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											fix_mistral_regex=True
											)
else:
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											)

tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)
base_model.eval()
print("Base model loaded successfully!")


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model loaded successfully!


## 2. Evaluate SFT Full-Information Model

In [ ]:
# Load the Full-Info LoRA adapter
try:
    print(f"Loading adapter from {SFT_FULL_DIR}...")
    model_full = PeftModel.from_pretrained(base_model, SFT_FULL_DIR)
    
    # Evaluate on Full Info Dataset
    acc_sft_full, results_sft_full = run_evaluation(
        model=model_full,
        tokenizer=tokenizer,
        dataset=val_full,
        training_strategy="DoRA",
        prompt_format="fullInfo",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    
    # Unload adapter to free memory for the next evaluation
    model_full.unload()
except Exception as e:
    print(f"Could not load or evaluate Full-Info SFT model: {e}")


## 3. Evaluate SFT Structural-Only Model

In [ ]:
# Load the Structural-Only LoRA adapter
try:
    print(f"Loading adapter from {SFT_STRUCT_DIR}...")
    model_struct = PeftModel.from_pretrained(base_model, SFT_STRUCT_DIR)
    
    # Evaluate on Structural Only Dataset
    acc_sft_struct, results_sft_struct = run_evaluation(
        model=model_struct,
        tokenizer=tokenizer,
        dataset=val_struct,
        training_strategy="DoRA",
        prompt_format="structOnly",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    
    # Unload adapter
    model_struct.unload()
except Exception as e:
    print(f"Could not load or evaluate Structural-Only SFT model: {e}")


## 4. Evaluate Dynamic Filtering (Top 4)

In [ ]:
# Evaluate Base Model on Filtered Dataset (StructOnly)
try:
    print("Evaluating Base Model with Dynamic Filtering (Top 4) - StructOnly...")
    run_evaluation(
        model=base_model,
        tokenizer=tokenizer,
        dataset=val_filtered_struct,
        training_strategy="Baseline - Dynamic Filtering (Top 4)",
        prompt_format="structOnly",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
except Exception as e:
    print(f"Could not evaluate Base model on filtered struct dataset: {e}")


In [ ]:
# Evaluate Base Model on Filtered Dataset (FullInfo)
try:
    print("Evaluating Base Model with Dynamic Filtering (Top 4) - FullInfo...")
    run_evaluation(
        model=base_model,
        tokenizer=tokenizer,
        dataset=val_filtered_full,
        training_strategy="Baseline - Dynamic Filtering (Top 4)",
        prompt_format="fullInfo",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
except Exception as e:
    print(f"Could not evaluate Base model on filtered full dataset: {e}")


In [7]:
# Evaluate SFT Model on Filtered Dataset (StructOnly)
try:
    print(f"Loading adapter from {SFT_STRUCT_DIR} for Filtered Eval...")
    model_struct_filtered = PeftModel.from_pretrained(base_model, SFT_STRUCT_DIR)
    
    print("Evaluating SFT Model with Dynamic Filtering (Top 4) - StructOnly...")
    run_evaluation(
        model=model_struct_filtered,
        tokenizer=tokenizer,
        dataset=val_filtered_struct,
        training_strategy="SFT LoRA - Dynamic Filtering (Top 4)",
        prompt_format="structOnly",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    model_struct_filtered.unload()
except Exception as e:
    print(f"Could not evaluate SFT model on filtered struct dataset: {e}")


Loading adapter from ..//output/models/granite-3.1-2b-instruct_structOnly_LoRA for Filtered Eval...
Evaluating SFT Model with Dynamic Filtering (Top 4) - StructOnly...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating SFT LoRA - Dynamic Filtering (Top 4) - structOnly:   0%|          | 0/2020 [00:00<?, ?it/s]/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Evaluating SFT LoRA - Dynamic Filtering (Top 4) - structOnly: 100%|██████████| 2020/2020 [03:41<00:00,  9.13it/s]

Logged new results to ..//artifacts/experiment_summary.csv
\n--- Evaluation Results ---
Training Strategy: SFT LoRA - Dynamic Filtering (Top 4)
Prompt Format: structOnly
Model: ibm-granite/granite-3.1-2b-instruct
Accuracy: 0.9782
Format Error Rate: 0.0000
Semantic Confusion: 0.5751
Option Bias (A): 0.4480
Latency: 221.37 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-3.1-2b-instruct_structOnly_SFT LoRA - Dynamic Filtering (Top 4).csv


In [8]:
# Evaluate SFT Model on Filtered Dataset (FullInfo)
try:
    print(f"Loading adapter from {SFT_FULL_DIR} for Filtered Eval...")
    model_full_filtered = PeftModel.from_pretrained(base_model, SFT_FULL_DIR)
    
    print("Evaluating SFT Model with Dynamic Filtering (Top 4) - FullInfo...")
    run_evaluation(
        model=model_full_filtered,
        tokenizer=tokenizer,
        dataset=val_filtered_full,
        training_strategy="SFT LoRA - Dynamic Filtering (Top 4)",
        prompt_format="fullInfo",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    model_full_filtered.unload()
except Exception as e:
    print(f"Could not evaluate SFT model on filtered full dataset: {e}")


Loading adapter from ..//output/models/granite-3.1-2b-instruct_fullInfo_LoRA for Filtered Eval...


Evaluating SFT Model with Dynamic Filtering (Top 4) - FullInfo...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating SFT LoRA - Dynamic Filtering (Top 4) - fullInfo:   0%|          | 0/2020 [00:00<?, ?it/s]/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Evaluating SFT LoRA - Dynamic Filtering (Top 4) - fullInfo: 100%|██████████| 2020/2020 [04:10<00:00,  8.07it/s]

Logged new results to ..//artifacts/experiment_summary.csv
\n--- Evaluation Results ---
Training Strategy: SFT LoRA - Dynamic Filtering (Top 4)
Prompt Format: fullInfo
Model: ibm-granite/granite-3.1-2b-instruct
Accuracy: 0.9594
Format Error Rate: 0.0000
Semantic Confusion: 0.5182
Option Bias (A): 0.4520
Latency: 250.18 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-3.1-2b-instruct_fullInfo_SFT LoRA - Dynamic Filtering (Top 4).csv
